# 03 - One-Class SVM Baseline

Trains a One-Class SVM on a 50,000-row stratified sample of CIC-IDS2017 (full dataset computationally infeasible for this algorithm).

**Approach:** unsupervised, distance-based boundary around normal data. Feature scaling applied (matters significantly here, unlike Isolation Forest, since the algorithm relies on distance calculations).

**Key result:** 37% precision / 44% recall on the ANOMALY class — underperforms Isolation Forest on both accuracy and training time (37 sec on 2% of the data vs. Isolation Forest's 9 sec on the full dataset).

### Load data and take a representative sample

In [2]:
import pandas as pd
df = pd.read_csv('/home/vboxuser/FEAR/data/cicids2017_cleaned.csv')
df_sample = df.sample(n=50000, random_state=42)
print(df_sample['Binary_Label'].value_counts())

Binary_Label
0    41724
1     8276
Name: count, dtype: int64


### Split features/labels and train/test

In [3]:
X = df_sample.drop(columns=['Label', 'Binary_Label'])
y = df_sample['Binary_Label']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

### Scale the features

In [4]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Train One-Class SVM

In [5]:
from sklearn.svm import OneClassSVM
import time

start = time.time()
oc_svm = OneClassSVM(kernel='rbf', nu=0.2, gamma='scale')
oc_svm.fit(X_train_scaled)
print(f'Training took {time.time() - start:.1f} seconds')

Training took 30.7 seconds


## Predict and evaluate

In [6]:
preds = oc_svm.predict(X_test_scaled)
preds_binary = [1 if p == -1 else 0 for p in preds]

from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_test, preds_binary))
print(classification_report(y_test, preds_binary))

[[10689  1828]
 [ 1396  1087]]
              precision    recall  f1-score   support

           0       0.88      0.85      0.87     12517
           1       0.37      0.44      0.40      2483

    accuracy                           0.79     15000
   macro avg       0.63      0.65      0.64     15000
weighted avg       0.80      0.79      0.79     15000



#### One-Class SVM: Full-Data Fairness Check (SGDOneClassSVM)

In [1]:
import pandas as pd
df = pd.read_csv('/home/vboxuser/FEAR/data/cicids2017_cleaned.csv')
print(df.shape)

(2520798, 80)


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import time

X_full = df.drop(columns=['Label', 'Binary_Label'])
y_full = df['Binary_Label']

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.3, random_state=42, stratify=y_full
)

scaler_full = StandardScaler()
X_train_scaled_full = scaler_full.fit_transform(X_train_full)
X_test_scaled_full = scaler_full.transform(X_test_full)
print(X_train_scaled_full.shape, X_test_scaled_full.shape)

(1764558, 78) (756240, 78)


In [4]:
from sklearn.linear_model import SGDOneClassSVM

start = time.time()
sgd_ocsvm = SGDOneClassSVM(nu=0.2, random_state=42)
sgd_ocsvm.fit(X_train_scaled_full)
print(f'Training took {time.time()-start:.1f} seconds')

Training took 3.1 seconds


In [5]:
preds_sgd = sgd_ocsvm.predict(X_test_scaled_full)
preds_sgd_binary = [1 if p == -1 else 0 for p in preds_sgd]

print(confusion_matrix(y_test_full, preds_sgd_binary))
print(classification_report(y_test_full, preds_sgd_binary))

[[609169  19349]
 [124403   3319]]
              precision    recall  f1-score   support

           0       0.83      0.97      0.89    628518
           1       0.15      0.03      0.04    127722

    accuracy                           0.81    756240
   macro avg       0.49      0.50      0.47    756240
weighted avg       0.71      0.81      0.75    756240



In [9]:
df_sample = df.sample(n=50000, random_state=42)

X = df_sample.drop(columns=['Label', 'Binary_Label'])
y = df_sample['Binary_Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.svm import OneClassSVM
oc_svm = OneClassSVM(kernel='rbf', nu=0.2, gamma='scale')
oc_svm.fit(X_train_scaled)

preds = oc_svm.predict(X_test_scaled)
preds_binary = [1 if p == -1 else 0 for p in preds]

In [10]:
original_labels_test_svm = df_sample.loc[y_test.index, 'Label']

results_svm = pd.DataFrame({
    'true_label': original_labels_test_svm.values,
    'predicted_anomaly': preds_binary
})

recall_by_type_svm = results_svm.groupby('true_label')['predicted_anomaly'].mean().sort_values()
print(recall_by_type_svm)

true_label
Bot                         0.000000
Web Attack � XSS            0.000000
Web Attack � Brute Force    0.000000
SSH-Patator                 0.000000
PortScan                    0.032907
FTP-Patator                 0.085714
BENIGN                      0.146041
DoS GoldenEye               0.475410
DDoS                        0.509960
DoS Hulk                    0.615779
DoS slowloris               0.647059
DoS Slowhttptest            0.937500
Name: predicted_anomaly, dtype: float64
